# TP 3 — La pente du Tonkpi par sous-préfecture

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP3_Tonkpi_pente_zonale.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 3 · KOUADIO Kouao Laurent

Objectifs :
1. Filtrer et joindre des couches vecteur avec **GeoPandas**.
2. Lire, assembler, masquer et reprojeter un **MNT** avec **rioxarray**.
3. Calculer une **pente** (opération focale).
4. Calculer des **statistiques zonales** par sous-préfecture.

Durée : environ 90 minutes. Connexion Internet nécessaire.
Lancez la cellule du MNT (partie 4) tôt : elle prend quelques minutes.

In [ ]:
# Google Colab : installe les paquets du cours (environ 1 minute).
# Dans l'environnement conda du cours (environment.yml), rien à faire.
import sys
if "google.colab" in sys.modules:
    %pip install -q pystac-client rioxarray

## 1. Imports

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import rioxarray
from pystac_client import Client
from rasterio.enums import Resampling
from rasterio.features import rasterize, shapes
from rioxarray.merge import merge_arrays
from shapely.geometry import shape

## 2. Lire les limites administratives

geoBoundaries donne les régions (ADM2) et les sous-préfectures (ADM3).
On garde une copie locale : si la connexion coupe, relancez la cellule.

In [ ]:
def geoboundaries(niveau):
    # télécharge une seule fois, puis lit la copie locale
    fichier = Path(f"civ_{niveau.lower()}.geojson")
    if not fichier.exists():
        url = ("https://www.geoboundaries.org/api/current/gbOpen/"
               "CIV/" + niveau + "/")
        meta = requests.get(url, timeout=60).json()
        r = requests.get(meta["gjDownloadURL"], timeout=600)
        r.raise_for_status()
        fichier.write_bytes(r.content)
    try:
        return gpd.read_file(fichier)
    except Exception:
        fichier.unlink()  # téléchargement incomplet : on recommence
        raise


adm2 = geoboundaries("ADM2")
adm3 = geoboundaries("ADM3")
print(len(adm2), "régions,", len(adm3), "sous-préfectures")
print(adm3.crs)
adm3.head()

**Véracité.** Regardez les noms : certains accents sont abîmés
(`DananÃ©` au lieu de `Danané`). Le fichier source a été encodé deux
fois. On répare les noms :

In [ ]:
def repare(nom):
    try:
        return nom.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return nom


for gdf in (adm2, adm3):
    gdf["shapeName"] = gdf["shapeName"].map(repare)
print([n for n in adm3["shapeName"] if n.startswith("Dan")])

## 3. Sélectionner le Tonkpi et ses sous-préfectures

In [ ]:
tonkpi = adm2[adm2["shapeName"] == "Tonkpi"]

# point intérieur : chaque sous-préfecture tombe dans une seule région
pts = adm3.copy()
pts["geometry"] = adm3.representative_point()
res = gpd.sjoin(pts, tonkpi[["geometry"]], predicate="within")
sp = adm3.loc[res.index].copy()
print(len(sp), "sous-préfectures dans le Tonkpi")

ax = adm2.plot(color="#EEF4F0", edgecolor="grey", lw=0.4,
               figsize=(6, 6))
sp.plot(ax=ax, color="#E07A1F", edgecolor="#1B4332", lw=0.5)
ax.set_axis_off()
plt.show()

**Question 1.** Refaites la jointure avec `predicate="intersects"` et
les polygones d'origine (sans `representative_point`). Combien de
sous-préfectures obtenez-vous ? Pourquoi ?

In [ ]:
for nom, crs in [("UTM 29N", "EPSG:32629"),
                 ("Albers Afrique", "ESRI:102022"),
                 ("Web Mercator", "EPSG:3857")]:
    km2 = tonkpi.to_crs(crs).area.iloc[0] / 1e6
    print(f"{nom:15s} {km2:10,.0f} km²")

**Question 2.** Pourquoi la surface en Web Mercator est-elle plus
grande ? Quel SCR utiliser pour mesurer des surfaces ?

## 4. Lire le MNT Copernicus GLO-30

Le MNT est découpé en tuiles de 1° × 1°, stockées en COG.

In [ ]:
bbox = [float(v) for v in tonkpi.total_bounds]
cat = Client.open("https://earth-search.aws.element84.com/v1")
items = cat.search(collections=["cop-dem-glo-30"],
                   bbox=bbox).item_collection()
print(len(items), "tuiles")

tuiles = []
for it in items:
    href = it.assets["data"].href.replace(
        "s3://copernicus-dem-30m/",
        "https://copernicus-dem-30m.s3.amazonaws.com/")
    da = rioxarray.open_rasterio(href, masked=True)
    tuiles.append(da.rio.clip_box(*bbox).load())
dem = merge_arrays(tuiles)
print(dem.shape, dem.rio.crs, dem.rio.resolution())

## 5. Masquer et reprojeter

In [ ]:
tk = dem.rio.clip(tonkpi.geometry, tonkpi.crs, drop=True)

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
dem.squeeze().plot.imshow(ax=axs[0], cmap="terrain")
axs[0].set_title("clip_box : rectangle")
tk.squeeze().plot.imshow(ax=axs[1], cmap="terrain")
axs[1].set_title("clip : polygone")
plt.show()

In [ ]:
demu = tk.rio.reproject("EPSG:32629", resolution=30,
                        resampling=Resampling.bilinear)
z = demu.squeeze().values.astype("float64")
print(demu.rio.crs, z.shape)
print(f"altitude : {np.nanmin(z):.0f} à {np.nanmax(z):.0f} m")

## 6. Calculer la pente

On calcule les dérivées par différences centrées sur la grille de
30 m, puis

$$\theta = \arctan\sqrt{(\partial z/\partial x)^2 +
(\partial z/\partial y)^2}.$$

In [ ]:
dzdy, dzdx = np.gradient(z, 30.0, 30.0)
pente = np.degrees(np.arctan(np.hypot(dzdx, dzdy)))
print(f"pente moyenne : {np.nanmean(pente):.1f}°")

plt.figure(figsize=(6, 6))
plt.imshow(pente, cmap="magma_r", vmin=0, vmax=35)
plt.colorbar(label="pente (degrés)", shrink=0.8)
plt.axis("off")
plt.show()

**Question 3.** Quelle part de la région a une pente supérieure à
20° ? Écrivez le code.

## 7. Statistiques zonales

On **rasterise** les sous-préfectures sur la grille du MNT, puis on
fait un `groupby` par numéro de zone.

In [ ]:
spu = sp.to_crs("EPSG:32629").reset_index(drop=True)
spu["zone"] = np.arange(1, len(spu) + 1)

zones = rasterize(zip(spu.geometry, spu["zone"]),
                  out_shape=z.shape,
                  transform=demu.rio.transform(), fill=0,
                  dtype="int32")
df = pd.DataFrame({"zone": zones.ravel(),
                   "pente": pente.ravel()}).dropna()
stats = (df[df["zone"] > 0].groupby("zone")["pente"].mean()
         .rename("pente_moy").reset_index())

# jointure attributaire : on rattache les statistiques aux polygones
spu = spu.merge(stats, on="zone")
spu.sort_values("pente_moy", ascending=False)[
    ["shapeName", "pente_moy"]].head(8).round(1)

In [ ]:
ax = spu.plot(column="pente_moy", cmap="YlOrBr", edgecolor="#1B4332",
              lw=0.5, legend=True, figsize=(6, 6),
              legend_kwds={"label": "pente moyenne (degrés)"})
ax.set_axis_off()
plt.show()

**Question 4.** Ajoutez l'altitude moyenne par sous-préfecture. Les
sous-préfectures les plus hautes sont-elles aussi les plus pentues ?

## 8. Bonus : vectoriser les zones raides

In [ ]:
raide = np.nan_to_num(pente) > 20
polys = [shape(g) for g, v in shapes(raide.astype("uint8"),
                                      mask=raide,
                                      transform=demu.rio.transform())
         if v == 1]
print(len(polys), "polygones,",
      round(sum(p.area for p in polys) / 1e6), "km²")

**Question 5 (bonus).** Pourquoi obtient-on autant de polygones ?
Comment réduire leur nombre (agrégation de pixels, surface minimale) ?

## À rendre

Le notebook exécuté avec vos réponses aux questions 1 à 5.